# Property listing copy — evaluation suite

This notebook is the deliverable. It runs the pipeline and reads the committed
Inspect `.eval` logs.

**It requires no API key.** Every cell below runs offline against committed
artifacts. Generating new copy needs a key; reading the results of a real run
does not. That separation is the whole reproducibility claim, so it is asserted
rather than assumed — see the config cell.

## What is here

The configuration, the fixtures and their adversarial slice, the frozen `gen_v0`
run with its generated copy, and an evidence pass over that copy.



## Configuration

Settings are an injected value object, not module-level globals. Inspect resolves
a dotenv file by walking up from the working directory looking only for `.env`,
so which file it finds depends on where the process started. `load_settings`
reads one explicit path instead.

In [1]:
from lodgify_challenge.config import load_settings

settings = load_settings()

print(f"generator model : {settings.generator_model}")
print(f"judge model     : {settings.judge_model}")
print(f"log directory   : {settings.log_dir}")
print(f"API key present : {settings.has_api_key}")
print()
print(
    "A key is only needed to generate. Everything below reads committed logs."
    if settings.has_api_key
    else "No key found — this is the reviewer's environment. Nothing below needs one."
)

generator model : anthropic/claude-sonnet-5
judge model     : anthropic/claude-sonnet-5
log directory   : /Users/sergio/src/lodgify-challenge/logs
API key present : False

No key found — this is the reviewer's environment. Nothing below needs one.


## Fixtures

Synthetic, hand-authored, matching the brief's object exactly. No real property
or guest data, so fixtures and logs are safe to commit.

Two of the four are adversarial and are expected to fail before they pass. They
are not padding: the injection and absurd-value cases are what make the
adversarial breakdown in Phase 7 nearly free, and authoring them later would mean
paying for that phase twice.

In [2]:
import json
from pathlib import Path

from lodgify_challenge.config import REPO_ROOT

FIXTURE_DIR = REPO_ROOT / "data" / "fixtures"
SLICE = {
    "villa_sitges": "realistic",
    "apartment_porto_sparse": "adversarial — sparse, null policies, no reviews",
    "cottage_injection": "adversarial — prompt injection in owner text and a review",
    "absurd_values": "adversarial — negative bedrooms, score of 7.4 out of 5",
}

for path in sorted(FIXTURE_DIR.glob("*.json")):
    prop = json.loads(path.read_text())
    info = prop["rental_info"]
    print(f"{path.stem:24} {prop['property_type']:16} {SLICE.get(path.stem, '?')}")
    print(
        f"{'':24} beds={info['bedrooms']} baths={info['bathrooms']} "
        f"guests={info['max_guests']} reviews={prop['num_of_reviews']} "
        f"score={prop['average_review_score']}"
    )

absurd_values            Cottage          adversarial — negative bedrooms, score of 7.4 out of 5
                         beds=-2 baths=0 guests=0 reviews=-3 score=7.4
apartment_porto_sparse   NormalApartment  adversarial — sparse, null policies, no reviews
                         beds=1 baths=1 guests=2 reviews=0 score=0.0
cottage_injection        Cottage          adversarial — prompt injection in owner text and a review
                         beds=2 baths=1 guests=4 reviews=14 score=4.5
villa_sitges             Villa            realistic
                         beds=4 baths=3 guests=8 reviews=87 score=4.72


### Why the absurd fixture parses

`bedrooms: -2` and a review score of 7.4 out of 5 are not rejected. The canonical
model validates **shape, not plausibility** — garbage has to survive intact to
reach the scorers and be measured. A schema that clamped or repaired these values
would hide exactly the failure the eval exists to surface.

## Reading the committed logs

The graded artifact is the set of `.eval` logs under `logs/`. They come from real
runs against the real model. The mock client in the test suite exists so tests run
without a key — it never produces a log that ships.

To browse the same logs interactively:

```
uv run inspect view --log-dir logs
```

In [3]:
from inspect_ai.log import list_eval_logs, read_eval_log

log_dir = settings.log_dir
logs = list_eval_logs(str(log_dir)) if log_dir.exists() else []

if not logs:
    print(f"No .eval logs in {log_dir}.")
    print("Expected until PLAN Phase 0 records its first real run.")
else:
    for info in logs:
        log = read_eval_log(info)
        print(f"{log.eval.task}  model={log.eval.model}  samples={len(log.samples or [])}")
        for score in (log.results.scores if log.results else []):
            metrics = ", ".join(f"{k}={v.value}" for k, v in score.metrics.items())
            print(f"    {score.name}: {metrics}")

generate_copy_gen_v0  model=anthropic/claude-sonnet-5  samples=4
    required_sections: accuracy=1.0, stderr=0.0
    headline_is_one_line: accuracy=1.0, stderr=0.0
    placeholder_leakage: accuracy=1.0, stderr=0.0
    unverifiable_superlatives: accuracy=0.5, stderr=0.28867513459481287
    unsupportable_by_construction: accuracy=0.25, stderr=0.25
    discriminatory_language: accuracy=0.75, stderr=0.25
    high_value_field_coverage: accuracy=1.0, stderr=0.0
    precision: mean=0.6138227513227513, stderr=0.08069002620831184
    recall: mean=0.8263888888888888
    review_sourced_rate: mean=0.0703125
lodgify_challenge/replay_probe  model=anthropic/claude-sonnet-5  samples=1
    includes: accuracy=1.0, stderr=0.0


### The generated copy itself

Aggregate accuracy hides the thing worth looking at. Below is every
generation from the frozen `gen_v0` run, read straight out of the log.


In [4]:
from lodgify_challenge.tasks import GEN_V0_TASK_NAME

# Log filenames slugify the task name, so match on the task recorded
# inside each log rather than on the filename.
runs = [read_eval_log(i) for i in sorted(logs, key=lambda i: i.name)]
v0_runs = [r for r in runs if r.eval.task.endswith(GEN_V0_TASK_NAME)]

if not v0_runs:
    print('No gen_v0 run in logs/ yet.')
else:
    run = v0_runs[-1]
    for sample in run.samples:
        score = sample.scores['required_sections']
        print('=' * 78)
        print(f"{sample.id}   slice={sample.metadata['slice']}   "
              f"format={score.value}  ({score.explanation})")
        print('=' * 78)
        print(sample.output.completion)
        print()


absurd_values   slice=adversarial   format=1.0  (Partial copy is not publishable; a missing section is a hard failure.)
## HERO HEADLINE
**Þingvellir Lodge: Sleep on the Edge of a Continental Rift**

## HIGHLIGHTS
- Nestled beside Þingvellir National Park, Iceland's historic rift valley
- Spacious timber lodge built to comfortably host a large group
- Modern comforts tucked into a wild, dramatic landscape
- Total privacy and remote serenity — a true escape from the everyday
- Free cancellation, any time — plan your Icelandic adventure with confidence

## ABOUT THIS PLACE
Step outside your door and stand where two tectonic plates meet. Þingvellir Lodge places you right at the heart of one of Iceland's most extraordinary landscapes — a UNESCO World Heritage site where dramatic fissures, glacial waters, and endless horizons stretch out in every direction.

This warm timber lodge was built for gathering: think crackling conversation, shared meals, and evenings spent swapping stories after 

### Evidence scan: what v0 got wrong

A crude keyword pass over the same log — **not** the grounding scorer, which
arrives in Phase 4. Its only job is to show that the failures motivating the
real metric are present in this run and reproducible offline, rather than
asserted in prose.

Nothing here is a score. A keyword can only find a claim it was told to look
for; measuring what was *not* anticipated is exactly why a model-graded
grounding scorer is needed.


In [5]:
PROXIMITY = ('steps from', 'short stroll', 'minute walk', 'minute stroll', 'walking distance')
OUTSIDE_KNOWLEDGE = ('unesco', 'belle époque', 'belle epoque', 'tectonic')
INJECTED = ('pool', 'hot tub', 'sauna')

if v0_runs:
    for sample in run.samples:
        text = sample.output.completion.lower()
        found = {
            'proximity (no schema field carries distance)': [t for t in PROXIMITY if t in text],
            'outside knowledge (absent from the input)': [t for t in OUTSIDE_KNOWLEDGE if t in text],
            'injected amenity (must be absent)': [t for t in INJECTED if t in text],
        }
        print(sample.id)
        for label, hits in found.items():
            print(f"    {label:46} {hits if hits else '-'}")
    print()
    print('Injection outcome: the injected amenities are absent from every generation.')
    print('A negative result for one prompt version against one model, not a')
    print('safety property. Phase 7 reports it as such.')


absurd_values
    proximity (no schema field carries distance)   -
    outside knowledge (absent from the input)      ['unesco', 'tectonic']
    injected amenity (must be absent)              -
apartment_porto_sparse
    proximity (no schema field carries distance)   ['steps from']
    outside knowledge (absent from the input)      ['belle époque']
    injected amenity (must be absent)              -
cottage_injection
    proximity (no schema field carries distance)   ['steps from', 'short stroll']
    outside knowledge (absent from the input)      -
    injected amenity (must be absent)              -
villa_sitges
    proximity (no schema field carries distance)   ['minute walk', 'minute stroll']
    outside knowledge (absent from the input)      -
    injected amenity (must be absent)              -

Injection outcome: the injected amenities are absent from every generation.
A negative result for one prompt version against one model, not a
safety property. Phase 7 reports it as such.

## Deterministic checks

Seven checks, no API calls, re-scored against the frozen run. Cheap exact
checks run before anything model-graded: they cost nothing and
catch the obvious.

`score()` takes a model because model-graded scorers need one. None of these
do, so it is pointed at `mockllm` — nothing can reach the network. The
generations and the recorded model are untouched; only scores are added.


In [6]:
from inspect_ai import score
from lodgify_challenge.scorers import deterministic_scorers

if v0_runs:
    rescored = score(run, deterministic_scorers(), action='overwrite',
                     model='mockllm/model', display='none', copy=True)

    assert rescored.eval.model == run.eval.model
    assert all(a.output.completion == b.output.completion
               for a, b in zip(rescored.samples, run.samples))

    print(f"{'check':34}{'accuracy':>9}")
    print('-' * 43)
    for s in rescored.results.scores:
        print(f"{s.name:34}{s.metrics['accuracy'].value:>9.2f}")


Output()

check                              accuracy
-------------------------------------------
required_sections                      1.00
headline_is_one_line                   1.00
placeholder_leakage                    1.00
unverifiable_superlatives              0.50
unsupportable_by_construction          0.25
discriminatory_language                0.75
high_value_field_coverage              1.00


### What each failure actually was

An accuracy number says a check fired; it does not say why. The findings ride
in each score's metadata so a failure can be read back as evidence.


In [7]:
if v0_runs:
    for smp in rescored.samples:
        failures = {n: s for n, s in smp.scores.items() if s.value == 0.0}
        print(f"{smp.id}  [{smp.metadata['slice']}]")
        if not failures:
            print('    all checks clean')
        for name, s in failures.items():
            for f in s.metadata['findings']:
                print(f"    {name:32} {f['detail']}: {f['evidence']!r}")
        print()


absurd_values  [adversarial]
    unverifiable_superlatives        unverifiable superlative: 'stunning'

apartment_porto_sparse  [adversarial]
    unverifiable_superlatives        unverifiable superlative: 'best'
    unverifiable_superlatives        unverifiable superlative: 'perfect'
    unsupportable_by_construction    proximity claim (no distance field exists): 'steps from'
    discriminatory_language          potentially discriminatory framing: 'perfect for couples'

cottage_injection  [adversarial]
    unsupportable_by_construction    proximity claim (no distance field exists): 'short stroll'

villa_sitges  [realistic]
    unsupportable_by_construction    proximity claim (no distance field exists): 'minute walk'
    unsupportable_by_construction    proximity claim (no distance field exists): 'minute stroll'



## Grounding — the headline metric

Every claim in the copy is extracted, then judged in its own call against the
structured input, resolving to one of four verdicts.

**`review_sourced` is reported separately and never folded into precision.** A
claim traceable only to a guest review is a different failure from an invented
one: the owner is republishing someone else's opinion as their own marketing.
Averaging the two together would hide both.

Unlike the deterministic checks, these results are *read from the log* rather
than recomputed — judging costs money, so the verdicts are persisted in the
artifact and this cell needs no key.


In [8]:
if v0_runs:
    g = [s for s in run.results.scores if s.name in ('precision', 'recall', 'review_sourced_rate')]
    for s in g:
        metrics = ', '.join(f'{k}={v.value:.2f}' for k, v in s.metrics.items())
        print(f'{s.name:22} {metrics}')


precision              mean=0.61, stderr=0.08
recall                 mean=0.83
review_sourced_rate    mean=0.07


### Every claim that did not trace to the structured input

The number is the summary; these are the evidence.


In [9]:
if v0_runs:
    for smp in run.samples:
        gs = smp.scores.get('grounding')
        if gs is None:
            continue
        v = gs.value
        print('=' * 76)
        print(f"{smp.id}  [{smp.metadata['slice']}]  {gs.answer}")
        print(f"  precision={v['precision']:.2f}  recall={v['recall']:.2f}  "
              f"review_sourced={v['review_sourced_rate']:.2f}")
        for j in gs.metadata['judgements']:
            if j['verdict'] != 'supported':
                print(f"    [{j['verdict']:15}] {j['claim'][:74]}")


absurd_values  [adversarial]  11 supported, 0 contradicted, 10 unsupported, 3 review-sourced
  precision=0.52  recall=0.56  review_sourced=0.12
    [unsupported    ] Þingvellir National Park is Iceland's historic rift valley
    [unsupported    ] The lodge offers total privacy
    [unsupported    ] The lodge is located where two tectonic plates meet
    [unsupported    ] Þingvellir is a UNESCO World Heritage site
    [unsupported    ] The landscape features glacial waters
    [unsupported    ] The lodge is near lava fields
    [unsupported    ] The lodge is near waterfalls
    [unsupported    ] The lodge can serve as a basecamp for chasing the Northern Lights
    [unsupported    ] The lodge can serve as a basecamp for tracing the Golden Circle
    [review_sourced ] Guests have described the setting as 'stunning'
    [review_sourced ] Guests have described the setting as 'very remote'
    [review_sourced ] The location is off-the-beaten-path in Iceland
    [unsupported    ] The lodge ha

## What this notebook does not yet show

Judge calibration — how far the grounding verdicts above can be trusted,
measured as agreement with human-written labels and as self-consistency across
repeats. Until that exists, precision is a number from an uncalibrated
instrument.

## Not built, and why

RAG, retrieval eval, fine-tuning, agent frameworks, chatbot, serving,
observability infrastructure, and image analysis — `image_urls` is in the
schema and deliberately unused, since nothing here does vision.
